In [1]:
import pandas as pd
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import plotly.express as px
import os
from dotenv import load_dotenv
from utils import convert_df_to_prophet, create_train_test, train_predict

load_dotenv()

/home/nbillard/miniconda3/envs/conda-jedha/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
df_rte = pd.read_csv(os.getenv('DATA_STORAGE') + 'power_consumption_2021_2025_utc.csv')
print(df_rte.shape)
df_rte.head()

(43074, 3)


,start_date,end_date,value
0,2021-01-01 00:00:00,2021-01-01 01:00:00,63955.5
1,2021-01-01 01:00:00,2021-01-01 02:00:00,62054.0
2,2021-01-01 02:00:00,2021-01-01 03:00:00,59021.5
3,2021-01-01 03:00:00,2021-01-01 04:00:00,57391.5
4,2021-01-01 04:00:00,2021-01-01 05:00:00,57620.5


In [3]:
dfp = convert_df_to_prophet(df_rte)
dfp.head()

,ds,y
0,2021-01-01 01:00:00,63955.5
1,2021-01-01 02:00:00,62054.0
2,2021-01-01 03:00:00,59021.5
3,2021-01-01 04:00:00,57391.5
4,2021-01-01 05:00:00,57620.5


### Train 1 year Test 1 day

In [33]:
dataset = dfp.copy()
dataset = dataset[dataset['ds'] < '2025-10-01']
train_period = "30 days"
test_period = "1 day"
period = "53 hours"
title = f"train_period: {train_period} - test_period: {test_period} - period: {period}"
train_test_sets = create_train_test(dataset, train_period, test_period, period=period, n_iter=5)

In [34]:
results = train_predict(train_test_sets)

17:01:21 - cmdstanpy - INFO - Chain [1] start processing
17:01:21 - cmdstanpy - INFO - Chain [1] done processing
17:01:21 - cmdstanpy - INFO - Chain [1] start processing
17:01:21 - cmdstanpy - INFO - Chain [1] done processing


SET 0
train from 2025-08-22 03:00:00 to 2025-09-21 02:00:00
test from 2025-09-21 03:00:00 to 2025-09-22 02:00:00
Train MAE: 1060.76, Test MAE: 1881.95
Train MAPE: 2.63, Test MAPE: 5.05
--------------------------------------------------
SET 1
train from 2025-08-24 08:00:00 to 2025-09-23 07:00:00
test from 2025-09-23 08:00:00 to 2025-09-24 07:00:00
Train MAE: 1038.13, Test MAE: 1338.96
Train MAPE: 2.56, Test MAPE: 2.87
--------------------------------------------------
SET 2
train from 2025-08-26 13:00:00 to 2025-09-25 12:00:00
test from 2025-09-25 13:00:00 to 2025-09-26 12:00:00


17:01:22 - cmdstanpy - INFO - Chain [1] start processing
17:01:22 - cmdstanpy - INFO - Chain [1] done processing
17:01:22 - cmdstanpy - INFO - Chain [1] start processing
17:01:22 - cmdstanpy - INFO - Chain [1] done processing
17:01:22 - cmdstanpy - INFO - Chain [1] start processing
17:01:22 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1054.40, Test MAE: 1464.78
Train MAPE: 2.59, Test MAPE: 2.92
--------------------------------------------------
SET 3
train from 2025-08-28 18:00:00 to 2025-09-27 17:00:00
test from 2025-09-27 18:00:00 to 2025-09-28 17:00:00
Train MAE: 1073.17, Test MAE: 2890.09
Train MAPE: 2.63, Test MAPE: 7.19
--------------------------------------------------
SET 4
train from 2025-08-30 23:00:00 to 2025-09-29 22:00:00
test from 2025-09-29 23:00:00 to 2025-09-30 22:00:00
Train MAE: 1157.83, Test MAE: 2005.55
Train MAPE: 2.82, Test MAPE: 4.49
--------------------------------------------------


In [29]:
# chart predict
def chart_predictions(df, results, title, display_period="30 days"):

    display_delta = pd.Timedelta(display_period)
    print(display_delta)
    last_date = df['ds'].max()
    display_start = last_date - display_delta
    display_mask = df['ds'] > display_start
    print(last_date, display_start)

    fig = px.line(df[display_mask], x='ds', y='y', range_y=[0, None], title=title)
    for i, res in enumerate(results):
        train_pred, test_pred = res
        fig.add_trace(px.scatter(test_pred, x='ds', y='yhat').data[0])
        fig.data[i+1].marker = dict(color='red')
    fig.show()

In [36]:
chart_predictions(dataset, results, title, "15 days")

15 days 00:00:00
2025-09-30 23:00:00 2025-09-15 23:00:00
